In [ ]:
from tqdm import tqdm
import datasets
import fev
import numpy as np
import spev

## Define FEV Task

In [ ]:
fev_task = fev.Task(
    dataset_path="autogluon/chronos_datasets",
    dataset_config="monash_car_parts",
    horizon=8,
    num_windows=1,
    seasonality=12,
    quantile_levels=[
        0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
        0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99
    ],
    target="target",
)

## Define SPEV Task

In [ ]:
from spev.task import Task

spev_task = Task(
    base=fev_task,
    aggregations=[
        {"lead_time": 0, "span": 1, "agg": "SUM"},
        {"lead_time": 2, "span": 1, "agg": "SUM"},
        {"lead_time": 0, "span": 8, "agg": "SUM"},
        # All,
    ],
    metrics=[
        {"name": "WCRPS", "epsilon": 0.01},
        {"name": "WQL", "quantile_level": 0.9, "epsilon": 0.01},
        {"name": "WQL", "quantile_level": 0.5, "epsilon": 0.01},
    ]
)

In [ ]:
spev_task.evaluation_summary(
    y_pred_per_window=[np.zeros(shape=(2674, 2501, 8, 1))],
)

In [ ]:
spev_task.evaluation_summary(
    y_pred_per_window=[np.ones(shape=(2674, 2501, 8, 1))],
)

## Forecast the FEV Task Using Chronos-2

In [ ]:
from spev.models import Chronos2Model

model = Chronos2Model(device="cpu")
chronos_predictions_for_fev = model.fit_predict(fev_task)

In [ ]:
print(chronos_predictions_for_fev)

## Turn Chronos-2 Quantile Forecasts into Sample Paths

In [ ]:
from spev.adapter import reshape_fev_predictions_per_window_to_sample_paths

sample_paths_per_window = reshape_fev_predictions_per_window_to_sample_paths(
    predictions_per_window=chronos_predictions_for_fev,
    task=fev_task,
    n_samples=2501,
    force_marginal=False,
)

## Evaluate Using SPEV Task

In [ ]:
spev_task.evaluation_summary(
    y_pred_per_window=sample_paths_per_window,
)